In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx
/kaggle/input/datasets/eslamfawzysayed/813-point/merged_file.csv


## Step 1: Install libraries

In [2]:
# ============================================================
# Step 1: Install libraries
# ============================================================

!pip install pymoo -q
!pip install openpyxl -q
!pip install gradio -q

print("✅ All libraries installed successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 1.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 41.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 1.9 MB/s eta 0:00:00eta 0:00:01
✅ All libraries installed successfully


# Step 2: Import libraries and load data

In [3]:

# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("🌱 Hydrochar Optimization System - Full Run")
print("=" * 70)

# Load data
file_path = '/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx'
df = pd.read_excel(file_path)
print(f"✅ Loaded {len(df)} rows of data")

# Define columns (as mentioned)
input_cols = [
    'C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 
    'Volatile matter%', 'Fixed carbon%', 
    'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)',
    'Temprature(°C)', 'Reaction Time (min)'
]

output_cols = [
    'yield(%)', 'Cc (%)', 'Hc(%)', 'Nc(%)', 'Sc(%)', 'Oc(%)',
    'HHVc (MJ/Kg)', 'EY(%)', 'ED(%)', 'CR(%)',
    'HC- Volatile matter%', 'HC- Fixed carbon%', 'HC- Ash(%)'
]

# Base columns (feedstock properties only)
base_input_cols = [
    'C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 
    'Volatile matter%', 'Fixed carbon%', 
    'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)'
]

# Filter existing columns
input_cols = [col for col in input_cols if col in df.columns]
output_cols = [col for col in output_cols if col in df.columns]
base_input_cols = [col for col in base_input_cols if col in df.columns]

print(f"📊 Inputs: {len(input_cols)} (11 properties + 2 conditions)")
print(f"📊 Outputs: {len(output_cols)} properties")
print(f"📊 Feedstock properties: {len(base_input_cols)} properties")

🌱 Hydrochar Optimization System - Full Run
✅ Loaded 271 rows of data
📊 Inputs: 13 (11 properties + 2 conditions)
📊 Outputs: 13 properties
📊 Feedstock properties: 11 properties


## Step 3: Train forward models (for prediction during optimization)

In [4]:
# ============================================================
# Step 3: Train forward models (for prediction during optimization)
# ============================================================

print("\n" + "=" * 70)
print("🔄 Training forward models (13 models)...")
print("=" * 70)

X = df[input_cols]  # 11 properties + temperature + time
Y = df[output_cols]  # 13 outputs

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

models_forward = {}

for i, output_col in enumerate(output_cols):
    print(f"  Training {i+1}/{len(output_cols)}: {output_col}")
    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, Y_train[output_col])
    models_forward[output_col] = model

print("✅ All forward models trained successfully")

# Min/max values for normalization
output_mins = {col: df[col].min() for col in output_cols}
output_maxs = {col: df[col].max() for col in output_cols}


🔄 Training forward models (13 models)...
  Training 1/13: yield(%)
  Training 2/13: Cc (%)
  Training 3/13: Hc(%)
  Training 4/13: Nc(%)
  Training 5/13: Sc(%)
  Training 6/13: Oc(%)
  Training 7/13: HHVc (MJ/Kg)
  Training 8/13: EY(%)
  Training 9/13: ED(%)
  Training 10/13: CR(%)
  Training 11/13: HC- Volatile matter%
  Training 12/13: HC- Fixed carbon%
  Training 13/13: HC- Ash(%)
✅ All forward models trained successfully


In [5]:
# ============================================================
# Step 4: Application criteria (according to the table)
# ============================================================

print("\n" + "=" * 70)
print("📊 Application criteria")
print("=" * 70)

application_criteria = {
    'Energy': {
        'name': 'Energy Production',
        'weights': {
            'yield(%)': 0.15, 'Cc (%)': 0.10, 'Hc(%)': 0.05,
            'Nc(%)': 0.05, 'Sc(%)': 0.05, 'Oc(%)': 0.05,
            'HHVc (MJ/Kg)': 0.25, 'EY(%)': 0.05, 'ED(%)': 0.05,
            'CR(%)': 0.05, 'HC- Volatile matter%': 0.05,
            'HC- Fixed carbon%': 0.10, 'HC- Ash(%)': 0.05
        },
        'targets': {
            'yield(%)': 'max', 'Cc (%)': 'max', 'Hc(%)': 'max',
            'Nc(%)': 'min', 'Sc(%)': 'min', 'Oc(%)': 'min',
            'HHVc (MJ/Kg)': 'max', 'EY(%)': 'max', 'ED(%)': 'max',
            'CR(%)': 'max', 'HC- Volatile matter%': 'min',
            'HC- Fixed carbon%': 'max', 'HC- Ash(%)': 'min'
        }
    },
    'Soil': {
        'name': 'Soil Amendment',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.25, 'Sc(%)': 0.20,
            'HC- Ash(%)': 0.30, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    },
    'Adsorbent': {
        'name': 'Adsorbent Material',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.20, 'Sc(%)': 0.15,
            'Oc(%)': 0.20, 'HC- Ash(%)': 0.20, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'Oc(%)': 'max', 'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    }
}

for app_name, criteria in application_criteria.items():
    print(f"\n🔹 {app_name}: {criteria['name']}")
    print(f"   Weights: {', '.join([f'{k}={v}' for k, v in criteria['weights'].items() if v > 0])}")


📊 Application criteria

🔹 Energy: Energy Production
   Weights: yield(%)=0.15, Cc (%)=0.1, Hc(%)=0.05, Nc(%)=0.05, Sc(%)=0.05, Oc(%)=0.05, HHVc (MJ/Kg)=0.25, EY(%)=0.05, ED(%)=0.05, CR(%)=0.05, HC- Volatile matter%=0.05, HC- Fixed carbon%=0.1, HC- Ash(%)=0.05

🔹 Soil: Soil Amendment
   Weights: yield(%)=0.15, Nc(%)=0.25, Sc(%)=0.2, HC- Ash(%)=0.3, HHVc (MJ/Kg)=0.1

🔹 Adsorbent: Adsorbent Material
   Weights: yield(%)=0.15, Nc(%)=0.2, Sc(%)=0.15, Oc(%)=0.2, HC- Ash(%)=0.2, HHVc (MJ/Kg)=0.1


## Step 5: Define NSGA-II problem

In [6]:
# ============================================================
# Step 5: Define NSGA-II problem
# ============================================================

print("\n" + "=" * 70)
print("🔄 Defining NSGA-II problem...")
print("=" * 70)

class OptimizationProblem(Problem):
    def __init__(self, raw_material, models_forward, input_cols, output_cols, 
                 weights, targets, df, output_mins, output_maxs):
        
        self.raw_material = raw_material
        self.models_forward = models_forward
        self.output_cols = output_cols
        self.weights = weights
        self.targets = targets
        self.output_mins = output_mins
        self.output_maxs = output_maxs
        
        # Temperature and time bounds
        temp_min = df['Temprature(°C)'].min()
        temp_max = df['Temprature(°C)'].max()
        time_min = df['Reaction Time (min)'].min()
        time_max = df['Reaction Time (min)'].max()
        
        xl = np.array([temp_min, time_min])
        xu = np.array([temp_max, time_max])
        
        # Active objectives (with weight > 0)
        self.active_outputs = [col for col, w in weights.items() if w > 0]
        n_obj = len(self.active_outputs)
        
        super().__init__(
            n_var=2,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            elementwise=False
        )
    
    def _evaluate(self, X, out, *args, **kwargs):
        n_solutions = X.shape[0]
        
        # Repeat raw material for each solution
        X_input = np.tile(self.raw_material, (n_solutions, 1))
        X_input = np.column_stack([X_input, X[:, 0], X[:, 1]])
        
        # Predict all outputs
        predictions = {}
        for col in self.output_cols:
            predictions[col] = self.models_forward[col].predict(X_input)
        
        # Calculate objectives
        F = np.zeros((n_solutions, self.n_obj))
        
        for i, output_col in enumerate(self.active_outputs):
            pred = predictions[output_col]
            min_val = self.output_mins[output_col]
            max_val = self.output_maxs[output_col]
            weight = self.weights[output_col]
            
            # Normalize
            if max_val > min_val:
                norm_val = (pred - min_val) / (max_val - min_val + 1e-10)
            else:
                norm_val = 0.5
            
            # Maximize or minimize
            if self.targets.get(output_col) == 'min':
                F[:, i] = norm_val * weight
            else:
                F[:, i] = -norm_val * weight
        
        out["F"] = F

print("✅ NSGA-II problem defined successfully")


🔄 Defining NSGA-II problem...
✅ NSGA-II problem defined successfully


In [7]:
!pip install pymoo -q

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
import warnings
import os
import joblib
import pickle
import json
from datetime import datetime
import time
import threading
import signal
import sys
import shutil

warnings.filterwarnings('ignore')

# ============================================================
# 1️⃣ TOPSIS SELECTION METHOD
# ============================================================

def topsis_selection(pareto_front, weights=None):
    """
    TOPSIS Method for selecting best solution from Pareto front
    """
    n_solutions, n_objectives = pareto_front.shape
    
    if weights is None:
        weights = np.ones(n_objectives) / n_objectives
    else:
        weights = np.array(weights) / np.sum(weights)
    
    # Normalization
    normalized = np.zeros_like(pareto_front)
    for j in range(n_objectives):
        col = pareto_front[:, j]
        norm_factor = np.sqrt(np.sum(col**2))
        if norm_factor > 0:
            normalized[:, j] = col / norm_factor
        else:
            normalized[:, j] = col
    
    # Weighted matrix
    weighted = normalized * weights
    
    # Ideal and Nadir solutions
    ideal_solution = np.min(weighted, axis=0)
    nadir_solution = np.max(weighted, axis=0)
    
    # Distances
    dist_ideal = np.sqrt(np.sum((weighted - ideal_solution)**2, axis=1))
    dist_nadir = np.sqrt(np.sum((weighted - nadir_solution)**2, axis=1))
    
    # TOPSIS score
    eps = 1e-10
    topsis_scores = dist_nadir / (dist_ideal + dist_nadir + eps)
    
    best_idx = np.argmax(topsis_scores)
    
    return best_idx, topsis_scores

# ============================================================
# 2️⃣ ADVANCED CHECKPOINT SYSTEM
# ============================================================

class AdvancedCheckpoint:
    def __init__(self, checkpoint_dir='checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.metadata_file = os.path.join(checkpoint_dir, 'metadata.json')
        
    def save_checkpoint(self, data, iteration, **kwargs):
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        checkpoint_file = os.path.join(
            self.checkpoint_dir, 
            f'checkpoint_iter_{iteration}_{timestamp}.pkl'
        )
        
        checkpoint_data = {
            'data': data,
            'iteration': iteration,
            'timestamp': timestamp,
            'kwargs': kwargs
        }
        
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint_data, f)
        
        self._update_metadata(iteration, checkpoint_file)
        
        if isinstance(data, list) and len(data) > 0:
            pd.DataFrame(data).to_csv(
                os.path.join(self.checkpoint_dir, f'data_backup_iter_{iteration}.csv'),
                index=False
            )
        
        return checkpoint_file
    
    def _update_metadata(self, iteration, checkpoint_file):
        metadata = {}
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, 'r') as f:
                metadata = json.load(f)
        
        metadata[str(iteration)] = {
            'file': checkpoint_file,
            'timestamp': datetime.now().isoformat(),
            'iteration': iteration
        }
        
        with open(self.metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
    
    def load_latest_checkpoint(self):
        if not os.path.exists(self.metadata_file):
            return None, 0
        
        with open(self.metadata_file, 'r') as f:
            metadata = json.load(f)
        
        if not metadata:
            return None, 0
        
        latest_iter = max(int(k) for k in metadata.keys())
        checkpoint_file = metadata[str(latest_iter)]['file']
        
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'rb') as f:
                checkpoint_data = pickle.load(f)
            return checkpoint_data['data'], latest_iter
        
        return None, 0

    def save_final_data(self, data, filename='final_optimal_data.csv'):
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df.to_csv(filename, index=False)
            df.to_excel(filename.replace('.csv', '.xlsx'), index=False)
            return df
        return None

# ============================================================
# 3️⃣ KAGGLE SESSION KEEPER
# ============================================================

class KaggleSessionKeeper:
    def __init__(self, interval=300):
        self.interval = interval
        self.running = True
        self.thread = threading.Thread(target=self._keep_alive, daemon=True)
        
    def start(self):
        self.running = True
        self.thread.start()
        
    def stop(self):
        self.running = False
        
    def _keep_alive(self):
        while self.running:
            time.sleep(self.interval)
            if self.running:
                if os.path.exists('final_optimal_data.csv'):
                    try:
                        temp_df = pd.read_csv('final_optimal_data.csv')
                        temp_df.to_csv('temp_keep_alive.csv', index=False)
                    except:
                        pass

# ============================================================
# 4️⃣ OPTIMIZATION PROBLEM CLASS (MODIFIED)
# ============================================================

class OptimizationProblemImproved(Problem):
    def __init__(self, raw_material, models_forward, input_cols, output_cols, 
                 weights, targets, df, output_mins, output_maxs):
        
        self.raw_material = raw_material
        self.models_forward = models_forward
        self.output_cols = output_cols
        self.weights = weights
        self.targets = targets
        self.output_mins = output_mins
        self.output_maxs = output_maxs
        
        temp_min = 170
        temp_max = 260
        time_min = 30
        time_max = 180
        
        xl = np.array([temp_min, time_min])
        xu = np.array([temp_max, time_max])
        
        self.active_outputs = [col for col, w in weights.items() if w > 0]
        n_obj = len(self.active_outputs)
        
        super().__init__(
            n_var=2,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            elementwise=False
        )
    
    def _evaluate(self, X, out, *args, **kwargs):
        n_solutions = X.shape[0]
        
        # Build full matrix (11 base features + Temperature + Time)
        X_input = np.tile(self.raw_material, (n_solutions, 1))
        X_input = np.column_stack([X_input, X[:, 0], X[:, 1]])  # 11 + 2 = 13 features
        
        predictions = {}
        for col in self.output_cols:
            if col in self.models_forward:
                predictions[col] = self.models_forward[col].predict(X_input)
            else:
                predictions[col] = np.zeros(n_solutions)
        
        F = np.zeros((n_solutions, self.n_obj))
        
        for i, output_col in enumerate(self.active_outputs):
            pred = predictions[output_col]
            min_val = self.output_mins.get(output_col, 0)
            max_val = self.output_maxs.get(output_col, 100)
            weight = self.weights[output_col]
            
            if max_val > min_val:
                norm_val = (pred - min_val) / (max_val - min_val + 1e-10)
            else:
                norm_val = 0.5
            
            # Use target to determine maximize or minimize
            if self.targets.get(output_col) == 'min':
                F[:, i] = -norm_val * weight  # Negative for minimization
            else:
                F[:, i] = norm_val * weight   # Positive for maximization
        
        out["F"] = F

# ============================================================
# 5️⃣ GENERATE OPTIMAL DATA WITH TOPSIS + CHECKPOINT
# ============================================================

def generate_optimal_data_with_topsis(df, models_forward, input_cols, output_cols,
                                     output_mins, output_maxs, application_criteria,
                                     base_input_cols, checkpoint_dir='checkpoints',
                                     save_interval=3):
    """
    Generate optimal data with TOPSIS and advanced checkpoint system
    """
    checkpoint_system = AdvancedCheckpoint(checkpoint_dir)
    
    all_data, start_iter = checkpoint_system.load_latest_checkpoint()
    if all_data is None:
        all_data = []
        start_iter = 0
    
    print(f"\n🔄 Starting from iteration: {start_iter}")
    print(f"📊 Current data points: {len(all_data)}")
    print(f"🎯 Selection method: TOPSIS")
    
    current_iter = start_iter
    
    try:
        for app_name, criteria in application_criteria.items():
            print(f"\n{'='*70}")
            print(f"📊 Generating data for application: {app_name} - {criteria['name']}")
            print(f"{'='*70}")
            
            weights = criteria['weights']
            targets = criteria['targets']
            
            weights_list = [weights[col] for col in weights.keys() if weights[col] > 0]
            
            for idx in range(len(df)):
                current_iter += 1
                
                if current_iter <= start_iter:
                    continue
                
                try:
                    raw_material = [df.iloc[idx][col] for col in base_input_cols]
                    
                    problem = OptimizationProblemImproved(
                        raw_material, models_forward, input_cols, output_cols,
                        weights, targets, df, output_mins, output_maxs
                    )
                    
                    algorithm = NSGA2(
                        pop_size=80,
                        sampling=FloatRandomSampling(),
                        crossover=SBX(prob=0.9, eta=15),
                        mutation=PM(prob=0.1, eta=20),
                        eliminate_duplicates=True
                    )
                    
                    res = minimize(
                        problem,
                        algorithm,
                        ('n_gen', 150),
                        seed=42 + idx,
                        verbose=False
                    )
                    
                    if len(res.X) > 0:
                        pareto_front = res.F
                        
                        # Use TOPSIS to select the best solution
                        best_idx, topsis_scores = topsis_selection(
                            pareto_front, 
                            weights=weights_list
                        )
                        
                        best_x = res.X[best_idx]
                        
                        temp, time_val = best_x
                        temp = np.clip(temp, 170, 260)
                        time_val = np.clip(time_val, 30, 180)
                        
                        X_input = np.array(raw_material + [temp, time_val]).reshape(1, -1)
                        
                        predictions = {}
                        for col in output_cols:
                            if col in models_forward:
                                predictions[col] = models_forward[col].predict(X_input)[0]
                            else:
                                predictions[col] = 0
                        
                        row = {
                            'Application': app_name,
                            'Temperature': temp,
                            'Time': time_val,
                            'Original_Sample_ID': idx,
                            'Iteration_Number': current_iter,
                            'TOPSIS_Score': topsis_scores[best_idx]
                        }
                        
                        for i, col in enumerate(base_input_cols):
                            row[col] = raw_material[i]
                        
                        for col in output_cols:
                            row[f'Pred_{col}'] = predictions.get(col, 0)
                        
                        all_data.append(row)
                        
                        # Save every save_interval points
                        if len(all_data) % save_interval == 0:
                            checkpoint_system.save_checkpoint(
                                all_data, 
                                current_iter,
                                app=app_name,
                                samples_processed=idx+1
                            )
                            print(f"💾 Checkpoint saved: {len(all_data)} points at iteration {current_iter}")
                    
                    if (idx + 1) % 10 == 0:
                        print(f"   ✅ Progress: {idx+1}/{len(df)} - Total points: {len(all_data)}")
                        
                except Exception as e:
                    print(f"⚠️ Error in sample {idx}: {e}")
                    checkpoint_system.save_checkpoint(
                        all_data, 
                        current_iter,
                        error=str(e),
                        app=app_name
                    )
                    continue
                
                if current_iter % 50 == 0:
                    checkpoint_system.save_checkpoint(
                        all_data, 
                        current_iter,
                        status='periodic_save'
                    )
                    print(f"⏰ Periodic checkpoint saved at iteration {current_iter}")
    
    except KeyboardInterrupt:
        print("\n⚠️ Process interrupted. Saving checkpoint...")
        checkpoint_system.save_checkpoint(all_data, current_iter, status='interrupted')
    except Exception as e:
        print(f"\n❌ Error: {e}. Saving emergency checkpoint...")
        checkpoint_system.save_checkpoint(all_data, current_iter, status='emergency', error=str(e))
        raise
    
    final_df = checkpoint_system.save_final_data(all_data)
    return final_df

# ============================================================
# 6️⃣ SPLIT DATA WITHOUT LEAKAGE
# ============================================================

def split_without_leakage(data, features_cols, target_cols, group_col='Original_Sample_ID', test_size=0.2):
    X = data[features_cols]
    y = data[target_cols]
    groups = data[group_col]
    
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
    print(f"\n📊 Data Split (No Leakage):")
    print(f"   Training: {len(X_train)} samples")
    print(f"   Testing: {len(X_test)} samples")
    
    return X_train, X_test, y_train, y_test

# ============================================================
# 7️⃣ TRAIN INVERSE MODEL
# ============================================================

def train_inverse_model(X_train, X_test, y_train, y_test):
    from sklearn.ensemble import RandomForestRegressor
    
    print("\n" + "=" * 70)
    print("🔄 TRAINING INVERSE MODEL")
    print("=" * 70)
    
    inverse_model = RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    
    inverse_model.fit(X_train, y_train)
    
    y_pred_train = inverse_model.predict(X_train)
    y_pred_test = inverse_model.predict(X_test)
    
    print(f"\n📊 Inverse Model Performance:")
    print(f"{'Target':<20} {'Train R²':<12} {'Test R²':<12}")
    print("-" * 50)
    
    for i, col in enumerate(y_train.columns):
        r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        
        status = "✅" if r2_test >= 0.7 else "⚠️"
        print(f"{col:<20} {r2_train:.3f}       {r2_test:.3f}       {status}")
    
    print("=" * 70)
    return inverse_model

# ============================================================
# 8️⃣ SIGNAL HANDLER
# ============================================================

def signal_handler(sig, frame):
    print('\n🛑 Saving emergency backup...')
    if 'all_data' in locals():
        try:
            pd.DataFrame(all_data).to_csv('emergency_backup.csv', index=False)
        except:
            pass
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

# ============================================================
# 🔟 MAIN EXECUTION - ALL APPLICATIONS
# ============================================================

print("\n" + "=" * 70)
print("🚀 STARTING OPTIMIZATION WITH TOPSIS + CHECKPOINT")
print("🎯 TARGETS: Energy, Soil, and Adsorbent")
print("=" * 70)

# ============================================================
# 📂 Load original file from Kaggle path
# ============================================================

file_name = '/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx'

print(f"\n📂 Loading file: {file_name}")

# Load file
try:
    if file_name.endswith('.csv'):
        df = pd.read_csv(file_name)
    else:
        df = pd.read_excel(file_name)
    print(f"✅ Loaded {len(df)} rows and {len(df.columns)} columns")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    sys.exit(1)

# ============================================================
# 📊 Display column names
# ============================================================

print("\n" + "=" * 70)
print("📊 DataFrame columns:")
print("=" * 70)
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")
print("=" * 70)

# ============================================================
# 📊 Define variables based on column names
# ============================================================

# Input columns (raw materials)
base_input_cols = ['C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 'Volatile matter%', 'Fixed carbon%', 'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)']

# Check if columns exist
missing_cols = [col for col in base_input_cols if col not in df.columns]
if missing_cols:
    print(f"⚠️ The following columns are missing: {missing_cols}")
    print("📌 Please update base_input_cols based on the column names shown above")

# Output columns (predicted results)
output_cols = ['yield(%)', 'Cc (%)', 'Hc(%)', 'Nc(%)', 'Sc(%)', 'Oc(%)', 'HHVc (MJ/Kg)', 'EY(%)', 'ED(%)', 'CR(%)', 'HC- Volatile matter%', 'HC- Fixed carbon%', 'HC- Ash(%)']

# Input columns (same as base_input_cols)
input_cols = base_input_cols.copy()

# Calculate min and max bounds for outputs
output_mins = {}
output_maxs = {}
for col in output_cols:
    if col in df.columns:
        output_mins[col] = df[col].min()
        output_maxs[col] = df[col].max()
    else:
        output_mins[col] = 0
        output_maxs[col] = 100

# ============================================================
# 🧠 Train forward models
# ============================================================

print("\n" + "=" * 70)
print("🧠 TRAINING FORWARD MODELS")
print("=" * 70)

from sklearn.ensemble import RandomForestRegressor

models_forward = {}

# Full features (11 base + Temperature + Time)
temp_col = 'Temprature(°C)' if 'Temprature(°C)' in df.columns else 'Temperature'
time_col = 'Reaction Time (min)' if 'Reaction Time (min)' in df.columns else 'Time'

full_features = base_input_cols + [temp_col, time_col]

# Check if required columns exist
available_features = [col for col in full_features if col in df.columns]
print(f"📊 Number of available features for training: {len(available_features)}")

# Prepare training data
X_train_forward = df[available_features].values

for col in output_cols:
    if col in df.columns:
        y_train_forward = df[col].values
        
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_forward, y_train_forward)
        models_forward[col] = model
        print(f"✅ Trained model for {col}")
    else:
        print(f"⚠️ Column {col} not found, skipping")

print("=" * 70)

# ============================================================
# ⚖️ Define application criteria - ALL APPLICATIONS
# ============================================================

print("\n" + "=" * 70)
print("📊 Application criteria")
print("=" * 70)

application_criteria = {
    'Energy': {
        'name': 'Energy Production',
        'weights': {
            'yield(%)': 0.15, 'Cc (%)': 0.10, 'Hc(%)': 0.05,
            'Nc(%)': 0.05, 'Sc(%)': 0.05, 'Oc(%)': 0.05,
            'HHVc (MJ/Kg)': 0.25, 'EY(%)': 0.05, 'ED(%)': 0.05,
            'CR(%)': 0.05, 'HC- Volatile matter%': 0.05,
            'HC- Fixed carbon%': 0.10, 'HC- Ash(%)': 0.05
        },
        'targets': {
            'yield(%)': 'max', 'Cc (%)': 'max', 'Hc(%)': 'max',
            'Nc(%)': 'min', 'Sc(%)': 'min', 'Oc(%)': 'min',
            'HHVc (MJ/Kg)': 'max', 'EY(%)': 'max', 'ED(%)': 'max',
            'CR(%)': 'max', 'HC- Volatile matter%': 'min',
            'HC- Fixed carbon%': 'max', 'HC- Ash(%)': 'min'
        }
    },
    'Soil': {
        'name': 'Soil Amendment',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.25, 'Sc(%)': 0.20,
            'HC- Ash(%)': 0.30, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    },
    'Adsorbent': {
        'name': 'Adsorbent Material',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.20, 'Sc(%)': 0.15,
            'Oc(%)': 0.20, 'HC- Ash(%)': 0.20, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'Oc(%)': 'max', 'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    }
}

for app_name, criteria in application_criteria.items():
    print(f"\n🔹 {app_name}: {criteria['name']}")
    print(f"   Weights: {', '.join([f'{k}={v}' for k, v in criteria['weights'].items() if v > 0])}")
    print(f"   Targets: {', '.join([f'{k}={v}' for k, v in criteria['targets'].items()])}")

print("=" * 70)

# ============================================================
# 🗑️ DELETE OLD CHECKPOINTS
# ============================================================

checkpoint_dirs = ['optimization_checkpoints', 'checkpoints']

for dir_name in checkpoint_dirs:
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)
        print(f"✅ Deleted: {dir_name}")

for file in os.listdir('.'):
    if file.endswith('.pkl') and 'checkpoint' in file:
        os.remove(file)
        print(f"✅ Deleted: {file}")

print("=" * 70)
print("🚀 STARTING FRESH - ALL APPLICATIONS FROM ITERATION 0")
print("📊 TARGETS: Energy, Soil, and Adsorbent")
print("=" * 70)

# Start Session Keeper
try:
    session_keeper = KaggleSessionKeeper(interval=240)
    session_keeper.start()
    print("✅ Session keeper started")
except:
    print("⚠️ Session keeper not available")

# ============================================================
# 🚀 Generate data for all applications - Currently disabled
# ============================================================

# ============================================================
# ⚠️ Data generation disabled - data already exists
# ============================================================

"""
# This section is disabled because data already exists in Excel file

optimal_data = generate_optimal_data_with_topsis(
    df, models_forward, input_cols, output_cols,
    output_mins, output_maxs, 
    application_criteria,
    base_input_cols,
    checkpoint_dir='optimization_checkpoints',
    save_interval=3
)

# ============================================================
# 💾 Save final results
# ============================================================

if optimal_data is not None and len(optimal_data) > 0:
    # Save complete data
    optimal_data.to_csv('optimal_data_all_applications.csv', index=False)
    optimal_data.to_excel('optimal_data_all_applications.xlsx', index=False)
    print(f"\n🎉 Completed! Number of points: {len(optimal_data)}")
    print("✅ Saved: optimal_data_all_applications.csv and optimal_data_all_applications.xlsx")
    
    # Display application distribution
    print("\n📊 Data distribution by application:")
    print(optimal_data['Application'].value_counts())
else:
    print("⚠️ No data generated!")
"""

print("\n" + "=" * 70)
print("⏭️ Skipping data generation - using existing file")
print("📂 Loading file from: /kaggle/input/datasets/eslamfawzysayed/813-point/merged_file.csv")
print("=" * 70)

# ============================================================
# 📂 Load merged file (existing data)
# ============================================================

file_merged = '/kaggle/input/datasets/eslamfawzysayed/813-point/merged_file.csv'

print(f"\n📂 Loading merged file: {file_merged}")

try:
    optimal_data = pd.read_csv(file_merged)
    print(f"✅ Loaded {len(optimal_data)} rows and {len(optimal_data.columns)} columns")
    print(f"\n📊 Available columns:")
    print(optimal_data.columns.tolist())
    
    # Display application distribution
    if 'Application' in optimal_data.columns:
        print(f"\n📊 Data distribution by application:")
        print(optimal_data['Application'].value_counts())
    else:
        print("\n⚠️ Application column not found in file")
        
except Exception as e:
    print(f"❌ Error loading file: {e}")
    optimal_data = None

# ============================================================
# 🔄 Train inverse model
# ============================================================

if optimal_data is not None and len(optimal_data) > 0:
    # Encode applications
    optimal_data_encoded = pd.get_dummies(optimal_data, columns=['Application'], prefix='Application')
    
    # Features
    app_cols = [col for col in optimal_data_encoded.columns if col.startswith('Application_')]
    feature_cols = base_input_cols + app_cols
    
    # Ensure all columns exist
    for col in feature_cols:
        if col not in optimal_data_encoded.columns:
            optimal_data_encoded[col] = 0
    
    # Targets
    target_cols = ['Temperature', 'Time'] + [col for col in optimal_data_encoded.columns if col.startswith('Pred_')]
    
    # Ensure required columns exist
    feature_cols = [col for col in feature_cols if col in optimal_data_encoded.columns]
    target_cols = [col for col in target_cols if col in optimal_data_encoded.columns]
    
    if len(target_cols) == 0:
        print("\n⚠️ No output columns (Pred_*) found in file!")
        print("   Will use Temperature and Time only")
        target_cols = ['Temperature', 'Time']
        target_cols = [col for col in target_cols if col in optimal_data_encoded.columns]
    
    print(f"\n📊 Features (X): {len(feature_cols)} columns")
    print(f"📊 Targets (y): {len(target_cols)} columns")
    print(f"\n📊 List of targets:")
    for i, col in enumerate(target_cols, 1):
        print(f"   {i}. {col}")

    X_train, X_test, y_train, y_test = split_without_leakage(
        optimal_data_encoded, feature_cols, target_cols,
        group_col='Original_Sample_ID', test_size=0.2
    )

    inverse_model = train_inverse_model(X_train, X_test, y_train, y_test)

    joblib.dump(inverse_model, 'inverse_model_all_applications.pkl')
    print("✅ Inverse model saved to 'inverse_model_all_applications.pkl'")
    
    # Save results
    from sklearn.metrics import r2_score, mean_squared_error
    y_pred_train = inverse_model.predict(X_train)
    y_pred_test = inverse_model.predict(X_test)
    
    results_df = pd.DataFrame()
    for i, col in enumerate(y_train.columns):
        results_df.loc[i, 'Target'] = col
        results_df.loc[i, 'R2_Train'] = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        results_df.loc[i, 'R2_Test'] = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        results_df.loc[i, 'RMSE_Test'] = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred_test[:, i]))
    
    results_df.to_csv('model_evaluation_all_applications.csv', index=False)
    print("✅ Saved evaluation to 'model_evaluation_all_applications.csv'")

# Stop Session Keeper
try:
    session_keeper.stop()
    print("⏹️ Session keeper stopped")
except:
    pass

print("\n" + "=" * 70)
print("🎉 OPTIMIZATION PIPELINE COMPLETED SUCCESSFULLY!")
print("📊 TARGETS: Energy, Soil, and Adsorbent")
print("=" * 70)


🚀 STARTING OPTIMIZATION WITH TOPSIS + CHECKPOINT
🎯 TARGETS: Energy, Soil, and Adsorbent

📂 Loading file: /kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx
✅ Loaded 271 rows and 26 columns

📊 DataFrame columns:
  1. C%
  2. H%
  3. N%
  4. S%
  5. O%
  6. Ash%
  7. Volatile matter%
  8. Fixed carbon%
  9. HHV (MJ/Kg)
  10. DM(%)
  11. water Content (%)
  12. Temprature(°C)
  13. Reaction Time (min)
  14. yield(%)
  15. Cc (%)
  16. Hc(%)
  17. Nc(%)
  18. Sc(%)
  19. Oc(%)
  20. HHVc (MJ/Kg)
  21. EY(%)
  22. ED(%)
  23. CR(%)
  24. HC- Volatile matter%
  25. HC- Fixed carbon%
  26. HC- Ash(%)

🧠 TRAINING FORWARD MODELS
📊 Number of available features for training: 13
✅ Trained model for yield(%)
✅ Trained model for Cc (%)
✅ Trained model for Hc(%)
✅ Trained model for Nc(%)
✅ Trained model for Sc(%)
✅ Trained model for Oc(%)
✅ Trained model for HHVc (MJ/Kg)
✅ Trained model for EY(%)
✅ Trained model for ED(%)
✅ Trained model for CR(%)
✅ Trained model for H

## 7️⃣ TRAIN INVERSE MODEL

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("🚀 Running Inverse Model on 15 Outputs")
print("=" * 80)

# ============================================================
# 1️⃣ Load Original File (for training forward models)
# ============================================================

file_original = "/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx"

print("\n📂 Loading original file...")
df_original = pd.read_excel(file_original)
print(f"✅ Loaded {len(df_original)} rows and {len(df_original.columns)} columns")

# ============================================================
# 2️⃣ Define Columns
# ============================================================

base_input_cols = ['C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 'Volatile matter%', 
                   'Fixed carbon%', 'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)']

output_cols = ['yield(%)', 'Cc (%)', 'Hc(%)', 'Nc(%)', 'Sc(%)', 'Oc(%)', 
               'HHVc (MJ/Kg)', 'EY(%)', 'ED(%)', 'CR(%)', 
               'HC- Volatile matter%', 'HC- Fixed carbon%', 'HC- Ash(%)']

temp_col = 'Temprature(°C)' if 'Temprature(°C)' in df_original.columns else 'Temperature'
time_col = 'Reaction Time (min)' if 'Reaction Time (min)' in df_original.columns else 'Time'

# ============================================================
# 3️⃣ Train Forward Models (to calculate Pred_*)
# ============================================================

print("\n" + "=" * 80)
print("🧠 Training Forward Models...")
print("=" * 80)

full_features = base_input_cols + [temp_col, time_col]
X_forward = df_original[full_features].values

models_forward = {}

for col in output_cols:
    if col in df_original.columns:
        y_forward = df_original[col].values
        
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_forward, y_forward)
        models_forward[col] = model
        print(f"✅ Trained model for {col}")
    else:
        print(f"⚠️ Column {col} not found")

print("=" * 80)

# ============================================================
# 4️⃣ Load 813-point file and calculate Pred_*
# ============================================================

print("\n" + "=" * 80)
print("📊 Calculating Pred_* for 813-point file...")
print("=" * 80)

file_813 = "/kaggle/input/datasets/eslamfawzysayed/813-point/merged_file.csv"
df_813 = pd.read_csv(file_813)
print(f"✅ Loaded {len(df_813)} rows")

# Calculate Pred_* for each row
X_813 = df_813[base_input_cols + ['Temperature', 'Time']].values

for col in output_cols:
    if col in models_forward:
        pred_col = f'Pred_{col}'
        df_813[pred_col] = models_forward[col].predict(X_813)
        print(f"✅ Calculated {pred_col}")

print(f"\n✅ File now contains {len(df_813.columns)} columns")

# ============================================================
# 5️⃣ Prepare Data for Inverse Model
# ============================================================

print("\n" + "=" * 80)
print("📊 Preparing data for Inverse Model...")
print("=" * 80)

# Encode Application
df_encoded = pd.get_dummies(df_813, columns=['Application'], prefix='App')
app_cols = [col for col in df_encoded.columns if col.startswith('App_')]

# Features (X)
feature_cols = base_input_cols + app_cols

# Targets (y) - all 15 outputs
target_cols = ['Temperature', 'Time'] + [f'Pred_{col}' for col in output_cols if col in models_forward]

# Ensure columns exist
feature_cols = [col for col in feature_cols if col in df_encoded.columns]
target_cols = [col for col in target_cols if col in df_encoded.columns]

X = df_encoded[feature_cols].astype(float)
y = df_encoded[target_cols].astype(float)

print(f"📌 Features (X): {X.shape}")
print(f"📌 Targets (y): {y.shape}")
print(f"\n📊 List of 15 outputs:")
for i, col in enumerate(target_cols, 1):
    print(f"   {i}. {col}")

# ============================================================
# 6️⃣ Split Data (No Leakage)
# ============================================================

def split_without_leakage(X, y, group_col='Original_Sample_ID', test_size=0.2):
    groups = df_encoded[group_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
    print(f"\n📊 Data Split:")
    print(f"   🟢 Training: {len(X_train)} samples")
    print(f"   🔵 Testing: {len(X_test)} samples")
    
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_without_leakage(X, y, test_size=0.2)

# ============================================================
# 7️⃣ Train Inverse Model (on 15 outputs)
# ============================================================

print("\n" + "=" * 80)
print("🧠 Training Inverse Model (15 outputs)...")
print("=" * 80)

# Train separate model for each output
final_models = {}
y_pred_train = np.zeros((len(X_train), y_train.shape[1]))
y_pred_test = np.zeros((len(X_test), y_test.shape[1]))

for i, col in enumerate(y_train.columns):
    model = RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train.iloc[:, i])
    final_models[col] = model
    y_pred_train[:, i] = model.predict(X_train)
    y_pred_test[:, i] = model.predict(X_test)
    print(f"✅ Trained {col}")

# ============================================================
# 8️⃣ Display Results (all 15 outputs)
# ============================================================

print("\n" + "=" * 80)
print("📊 Inverse Model Performance (15 outputs):")
print("=" * 80)
print(f"{'Output':<40} {'Train R²':<15} {'Test R²':<15} {'RMSE':<15}")
print("-" * 85)

results = {}
for i, col in enumerate(y_train.columns):
    r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
    r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
    rmse_test = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred_test[:, i]))
    
    results[col] = {
        'r2_train': r2_train,
        'r2_test': r2_test,
        'rmse_test': rmse_test
    }
    
    status = "✅" if r2_test >= 0.7 else "⚠️" if r2_test >= 0.5 else "❌"
    print(f"{col:<40} {r2_train:.4f}       {r2_test:.4f}       {rmse_test:.4f}       {status}")

print("=" * 85)

# ============================================================
# 9️⃣ Summary Statistics
# ============================================================

print("\n" + "=" * 80)
print("📊 Summary Statistics:")
print("=" * 80)

r2_train_values = [results[col]['r2_train'] for col in results]
r2_test_values = [results[col]['r2_test'] for col in results]

print(f"   📌 Number of outputs: {len(results)}")
print(f"\n   📈 Train R²:")
print(f"      Mean: {np.mean(r2_train_values):.4f}")
print(f"      Best: {np.max(r2_train_values):.4f}")
print(f"      Worst: {np.min(r2_train_values):.4f}")
print(f"\n   📈 Test R²:")
print(f"      Mean: {np.mean(r2_test_values):.4f}")
print(f"      Best: {np.max(r2_test_values):.4f}")
print(f"      Worst: {np.min(r2_test_values):.4f}")
print(f"\n   ✅ R² > 0.8: {sum(1 for v in r2_test_values if v > 0.8)} outputs")
print(f"   ✅ R² > 0.7: {sum(1 for v in r2_test_values if v > 0.7)} outputs")
print(f"   ✅ R² > 0.6: {sum(1 for v in r2_test_values if v > 0.6)} outputs")

# ============================================================
# 🔟 Save Model
# ============================================================

print("\n" + "=" * 80)
print("💾 Saving Model...")
print("=" * 80)

joblib.dump(final_models, '/kaggle/working/inverse_model_15_outputs.pkl')
print("✅ Model saved to: inverse_model_15_outputs.pkl")

# Save evaluation results
results_df = pd.DataFrame([
    {
        'Output': col,
        'R2_Train': results[col]['r2_train'],
        'R2_Test': results[col]['r2_test'],
        'RMSE_Test': results[col]['rmse_test']
    }
    for col in results
])
results_df.to_csv('/kaggle/working/model_evaluation_15_outputs.csv', index=False)
print("✅ Evaluation saved to: model_evaluation_15_outputs.csv")

print("\n" + "=" * 80)
print("🎉 Process completed successfully!")
print(f"📊 Number of outputs: {len(results)}")
print("=" * 80)

🚀 Running Inverse Model on 15 Outputs

📂 Loading original file...
✅ Loaded 271 rows and 26 columns

🧠 Training Forward Models...
✅ Trained model for yield(%)
✅ Trained model for Cc (%)
✅ Trained model for Hc(%)
✅ Trained model for Nc(%)
✅ Trained model for Sc(%)
✅ Trained model for Oc(%)
✅ Trained model for HHVc (MJ/Kg)
✅ Trained model for EY(%)
✅ Trained model for ED(%)
✅ Trained model for CR(%)
✅ Trained model for HC- Volatile matter%
✅ Trained model for HC- Fixed carbon%
✅ Trained model for HC- Ash(%)

📊 Calculating Pred_* for 813-point file...
✅ Loaded 813 rows
✅ Calculated Pred_yield(%)
✅ Calculated Pred_Cc (%)
✅ Calculated Pred_Hc(%)
✅ Calculated Pred_Nc(%)
✅ Calculated Pred_Sc(%)
✅ Calculated Pred_Oc(%)
✅ Calculated Pred_HHVc (MJ/Kg)
✅ Calculated Pred_EY(%)
✅ Calculated Pred_ED(%)
✅ Calculated Pred_CR(%)
✅ Calculated Pred_HC- Volatile matter%
✅ Calculated Pred_HC- Fixed carbon%
✅ Calculated Pred_HC- Ash(%)

✅ File now contains 29 columns

📊 Preparing data for Inverse Model...
📌 

# 🖥️ # Step 9: Interactive Dashboard

In [ ]:
# ================================================.txt
# ============================================================
# 🌱 Hydrochar from Digestat AI System - MODERN DESIGN
# ============================================================

!pip install gradio openpyxl -q

import gradio as gr
import numpy as np
import pandas as pd
import joblib
import datetime
import traceback

# ============================================================
# 🔧 TRY TO IMPORT SPACES (only works on Hugging Face)
# ============================================================

try:
    import spaces
    SPACES_AVAILABLE = True
    print("✅ Running on Hugging Face Spaces with GPU support")
except ImportError:
    SPACES_AVAILABLE = False
    print("ℹ️ Running locally (Kaggle/Colab) - GPU not available")
    
    class DummySpaces:
        @staticmethod
        def GPU(func):
            return func
    
    spaces = DummySpaces()

# ============================================================
# LOAD MODEL - 
# ============================================================

model_path = "/kaggle/working/inverse_model_15_outputs.pkl"

try:
    model = joblib.load(model_path)
    print("✅ Model loaded successfully from:", model_path)
    print("   Model type:", type(model))
    
    # Test the model
    if isinstance(model, dict):
        print(f"   The model is a dictionary containing {len(model)} sub-models")
        print(f"   Outputs: {list(model.keys())}")
    else:
        print("   Model type:", type(model))
        
except Exception as e:
    print(f"⚠️ Could not load model from {model_path}")
    print(f"   Error: {e}")
    print("   Using dummy model for testing...")
    
    class DummyModel:
        def predict(self, X):
            return np.array([[
                220, 180, 72, 68, 5, 1.8, 0.4, 22, 28, 75, 35, 80, 40, 50, 8
            ]])
    model = DummyModel()

# ============================================================
# OUTPUT NAMES - 15
# ============================================================

output_names = [
    "Temperature (°C)",
    "Residence Time (min)",
    "Yield (%)",
    "Cc (%)",
    "Hc (%)",
    "Nc (%)",
    "Sc (%)",
    "Oc (%)",
    "HHVc (MJ/Kg)",
    "EY (%)",
    "ED (%)",
    "CR (%)",
    "HC- Volatile matter (%)",
    "HC- Fixed carbon (%)",
    "HC- Ash (%)"
]

# ============================================================
# PREDICTION FUNCTION WITH ERROR HANDLING
# ============================================================

@spaces.GPU
def prediction(
    C, H, N, S, O,
    Ash, VM, FC, HHV,
    DM, Water, Application
):
    try:
        # 
        app = [
            1 if Application == "Energy" else 0,
            1 if Application == "Soil" else 0,
            1 if Application == "Adsorbent" else 0
        ]
        
        # 
        X = np.array([
            C, H, N, S, O,
            Ash, VM, FC, HHV,
            DM, Water, *app
        ]).reshape(1, -1)
        
        print(f"🔍 Shape of X: {X.shape}")
        print(f"🔍 X values: {X[0]}")
        
        # Prediction
        if isinstance(model, dict):
            # 
            y = np.array([model[col].predict(X)[0] for col in model.keys()])
        else:
            # 
            y = model.predict(X)[0]
        
        print(f"🔍 Predictions: {y}")
        
        # 
        if len(y) < 15:
            print(f"⚠️ Model returned {len(y)} outputs only, expected 15")
            # 
            y = list(y) + [0] * (15 - len(y))
            y = np.array(y)
        
        # HTML 
        html = """
        <style>
            .result-container {
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 12px 15px;
                border-radius: 12px;
                box-shadow: 0 8px 30px rgba(102, 126, 234, 0.3);
                margin: 0px;
            }
            .result-title {
                color: white;
                font-size: 15px;
                font-weight: bold;
                text-align: center;
                margin-bottom: 10px;
                text-shadow: 0 2px 4px rgba(0,0,0,0.2);
                letter-spacing: 0.5px;
            }
            .result-grid {
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 8px;
                background: rgba(255,255,255,0.95);
                border-radius: 10px;
                padding: 10px 12px;
                box-shadow: 0 4px 12px rgba(0,0,0,0.08);
            }
            .result-item {
                display: flex;
                justify-content: space-between;
                padding: 5px 8px;
                border-bottom: 1px solid #f0f0f0;
                transition: all 0.3s ease;
                border-radius: 4px;
            }
            .result-item:hover {
                background: linear-gradient(90deg, #f8f9fa 0%, #e9ecef 100%);
                transform: scale(1.02);
            }
            .result-item .property {
                font-size: 12px;
                color: #2d3748;
                font-weight: 500;
            }
            .result-item .value {
                font-size: 13px;
                font-weight: bold;
                color: #667eea;
            }
            .footer-note {
                text-align: center;
                color: rgba(255,255,255,0.8);
                font-size: 11px;
                margin-top: 8px;
            }
            .result-item:last-child {
                border-bottom: none;
            }
            .result-item:nth-last-child(2) {
                border-bottom: none;
            }
        </style>
        
        <div class="result-container">
            <div class="result-title">
                📊 Hydrochar Properties
            </div>
            <div class="result-grid">
        """
        
        # Display all 15 outputs
        for i, (p, v) in enumerate(zip(output_names, y[:15])):
            html += f"""
                <div class="result-item">
                    <span class="property">{p}</span>
                    <span class="value">{v:.3f}</span>
                </div>
            """
        
        html += """
            </div>
            <div class="footer-note">
                ⚡ Powered by Random Forest AI Model (15 Outputs)
            </div>
        </div>
        """
        
        return html
        
    except Exception as e:
        # 
        error_msg = traceback.format_exc()
        print(f"❌ Error in prediction: {error_msg}")
        
        return f"""
        <div style="background: #fee; padding: 20px; border-radius: 12px; border: 2px solid #f44;">
            <h3 style="color: #d00;">❌ Prediction Error</h3>
            <p><strong>Error:</strong> {str(e)}</p>
            <p><strong>X shape:</strong> {X.shape if 'X' in locals() else 'Unknown'}</p>
            <p><strong>Model type:</strong> {type(model)}</p>
            <details>
                <summary>Details</summary>
                <pre style="background: #f5f5f5; padding: 10px; border-radius: 5px; font-size: 11px; max-height: 300px; overflow: auto;">{error_msg}</pre>
            </details>
        </div>
        """

# ============================================================
# CSS - DESIGN MODERN 
# ============================================================

css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%) !important;
    padding: 20px !important;
    border-radius: 20px !important;
    border: 4px solid #667eea !important;
    box-shadow: 0 0 0 8px #764ba2, 0 20px 60px rgba(0,0,0,0.15) !important;
}

.header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    padding: 10px 20px !important;
    border-radius: 10px !important;
    text-align: center !important;
    margin-bottom: 0px !important;
    box-shadow: 0 6px 20px rgba(102, 126, 234, 0.2) !important;
    border: none !important;
}

.header h1 {
    font-size: 18px !important;
    color: white !important;
    margin: 0 !important;
    font-weight: 600 !important;
    text-shadow: 0 2px 4px rgba(0,0,0,0.15) !important;
    letter-spacing: 0.5px !important;
}

.box {
    background: rgba(255,255,255,0.95) !important;
    padding: 12px !important;
    border-radius: 12px !important;
    border: 2px solid #e2e8f0 !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.08) !important;
    margin-top: 0px !important;
    transition: all 0.3s ease !important;
}

.box:hover {
    box-shadow: 0 12px 40px rgba(0,0,0,0.12) !important;
    transform: translateY(-2px) !important;
}

.box h3, .box .gr-markdown {
    display: none !important;
}

input, .gr-number input, .gr-textbox input {
    height: 32px !important;
    font-size: 11px !important;
    border: 2px solid #e2e8f0 !important;
    border-radius: 8px !important;
    background: #f7fafc !important;
}

input:focus, .gr-number input:focus {
    border-color: #667eea !important;
    box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.2) !important;
    background: white !important;
    outline: none !important;
}

select, .gr-dropdown select {
    height: 32px !important;
    font-size: 11px !important;
    border: 2px solid #e2e8f0 !important;
    border-radius: 8px !important;
    background: linear-gradient(135deg, #f7fafc 0%, #edf2f7 100%) !important;
}

select:focus {
    border-color: #667eea !important;
    box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.2) !important;
}

button {
    height: 36px !important;
    font-size: 14px !important;
    font-weight: 600 !important;
    border: none !important;
    border-radius: 10px !important;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 4px 15px rgba(102, 126, 234, 0.3) !important;
    letter-spacing: 0.5px !important;
}

button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(102, 126, 234, 0.4) !important;
}

.tabs {
    display: none !important;
}

label, .gr-label {
    font-weight: 500 !important;
    color: #2d3748 !important;
    font-size: 11px !important;
    margin-bottom: 2px !important;
}

.gr-file, .download-btn, .file-download {
    display: none !important;
}

.gr-row {
    gap: 12px !important;
}

.gr-box {
    margin-top: 0px !important;
    padding-top: 0px !important;
}

.gr-row .gr-column {
    padding-top: 0px !important;
}

.gr-row .gr-column .gr-row {
    gap: 6px !important;
}

.gr-row .gr-column .gr-column {
    padding: 0px 4px !important;
}

.footer, .gradio-footer, .gradio-footer-version,
.built-with, .built-with-gradio,
.app-footer, .footer-text,
.parameters, .api-parameters,
.utilizer, .utilizer-text,
[class*="utilizer"], [class*="Parameters"],
[class*="api"], [class*="API"] {
    display: none !important;
}

div:has(> span:contains("Built with Gradio")),
div:has(> span:contains("utlizer")),
div:has(> span:contains("Parameters")),
div:has(> span:contains("API")) {
    display: none !important;
}

.dashboard-frame {
    border: 3px solid #667eea !important;
    border-radius: 20px !important;
    padding: 5px !important;
    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%) !important;
    box-shadow: 0 0 30px rgba(102, 126, 234, 0.3) !important;
}
"""

# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    css=css,
    title="🌱 Hydrochar from Digestat",
    elem_classes="dashboard-frame"
) as demo:
    
    gr.HTML("""
    <div class="header">
        <h1>🌱 Hydrochar from Digestat</h1>
    </div>
    """)
    
    with gr.Row():
        with gr.Column(scale=1, elem_classes="box"):
            with gr.Row():
                with gr.Column(scale=1, min_width=60):
                    C = gr.Number(label="🟢 Carbon C (%)", value=45)
                    H = gr.Number(label="🔵 Hydrogen H (%)", value=6)
                    N = gr.Number(label="🟡 Nitrogen N (%)", value=1.2)
                with gr.Column(scale=1, min_width=60):
                    S = gr.Number(label="🟠 Sulfur S (%)", value=0.5)
                    O = gr.Number(label="🔴 Oxygen O (%)", value=38)
                    Ash = gr.Number(label="⚪ Ash (%)", value=8)
                with gr.Column(scale=1, min_width=60):
                    VM = gr.Number(label="🔥 Volatile Matter (%)", value=50)
                    FC = gr.Number(label="⬛ Fixed Carbon (%)", value=40)
                    HHV = gr.Number(label="⚡ HHV (MJ/kg)", value=20)
                with gr.Column(scale=1, min_width=60):
                    DM = gr.Number(label="💧 Dry Matter (%)", value=90)
                    Water = gr.Number(label="💦 Water (%)", value=10)
                    Application = gr.Dropdown(
                        ["Energy", "Soil", "Adsorbent"],
                        value="Energy",
                        label="🎯 Application"
                    )
            
            predict_btn = gr.Button("🚀 Generate Prediction", variant="primary")
        
        with gr.Column(scale=1, elem_classes="box"):
            output = gr.HTML()
    
    predict_btn.click(
        prediction,
        inputs=[C, H, N, S, O, Ash, VM, FC, HHV, DM, Water, Application],
        outputs=[output]
    )

# ============================================================
# RUN
# ============================================================

demo.launch(share=True, debug=True)  # debug=True 

# ============================================================
# ℹ️ ============================================================

print("\n" + "="*60)
print("📸 TO CAPTURE SCREENSHOT:")
print("="*60)
print("1. Wait for the Gradio link to appear above")
print("2. Copy the public URL (starts with https://xxxxx.gradio.live)")
print("3. Open it in your browser")
print("4. Take screenshot using browser tools (F12 → Screenshot)")
print("="*60)

ℹ️ Running locally (Kaggle/Colab) - GPU not available
✅ Model loaded successfully from: /kaggle/working/inverse_model_15_outputs.pkl
   Model type: <class 'dict'>
   The model is a dictionary containing 15 sub-models
   Outputs: ['Temperature', 'Time', 'Pred_yield(%)', 'Pred_Cc (%)', 'Pred_Hc(%)', 'Pred_Nc(%)', 'Pred_Sc(%)', 'Pred_Oc(%)', 'Pred_HHVc (MJ/Kg)', 'Pred_EY(%)', 'Pred_ED(%)', 'Pred_CR(%)', 'Pred_HC- Volatile matter%', 'Pred_HC- Fixed carbon%', 'Pred_HC- Ash(%)']
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d2c4e2aca931cabeff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔍 Shape of X: (1, 14)
🔍 X values: [45.   6.   1.2  0.5 38.   8.  50.  40.  20.  90.  10.   1.   0.   0. ]
🔍 Predictions: [215.17599049  98.47310591  68.69334544  58.36394871   5.93496337
   1.49526089   0.39492837  28.16957949  24.19387743  79.72855267
   1.06133823  77.42867553  47.77031614  35.66110513   7.56031343]
